# 00 Walk-Forward Backtest

A single 80/20 train/test split flatters every model in a time-series setting. Walk-forward backtesting is the honest version: re-fit the model at every point in time and accumulate genuine pseudo-out-of-sample errors.


## Table of Contents
- [The problem with a single split](#single-split)
- [Walk-forward, mechanically](#walk-forward)
- [Backtest OLS on GDP growth](#bt-ols)
- [Backtest Random Forest and XGBoost](#bt-trees)
- [Compare models honestly](#compare)
- [Errors over time](#errors-over-time)
- [Simple ensemble](#ensemble)
- [Checkpoint (Self-Check)](#checkpoint-self-check)
- [Solutions (Reference)](#solutions-reference)


## Why This Notebook Matters
By §05 you have run OLS, Random Forest, and XGBoost on the same target with the same features. You probably picked a winner based on a single train/test split. That number is **almost certainly too optimistic**, because the test window happens to be one specific period and the model was tuned to avoid embarrassment in exactly that period.

Walk-forward backtesting answers the harder question: *if I had been deploying this model in real time from 1995 to today, what error would I actually have realized?* The answer is what you should report, not the single-split number.

You will learn:
- why single train/test splits flatter time-series models,
- how to implement an expanding-window walk-forward backtest from scratch,
- how to apply it to OLS, Random Forest, and XGBoost on the same target,
- how to read forecast errors **over time** instead of as a single aggregate,
- how a naive average of model forecasts can beat any individual model.

## Prerequisites (Quick Self-Check)
- §01 (data) — you have the quarterly macro panel.
- §01b (stationarity) — you understand why the target is a growth rate, not a level.
- §02 (regression) and §02b (ML regression) — you have fit OLS / RF / XGBoost at least once.
- §03 (classification) — you saw walk-forward applied to recession classification.

## What You Will Produce
- A DataFrame of pseudo-out-of-sample predictions and realized values for OLS, RF, and XGBoost.
- A comparison table of OOS RMSE / MAE / R² for each model and a simple ensemble.
- A plot of forecast errors over time that shows when each model fails.

## Common Pitfalls
- Picking the test window to make the model look good. Use a fixed protocol you commit to before looking at results.
- Re-tuning hyperparameters on the test window. The whole point of OOS evaluation is that the test window is *unseen* until the very end.
- Confusing expanding-window with rolling-window. Both are valid; pick one consciously.
- Reporting a single aggregate number. Time-series of errors is much more informative.

## Quick Fixes (When You Get Stuck)
- If walk-forward is slow, reduce the number of refits by widening the step size.
- If the first few predictions look wild, your initial training window is too small. Bump it up.
- If predictions and realized values are misaligned, check that you used `shift(-1)` or equivalent to build the target *and* did not leak it into features.

## Matching Guide
- `docs/guides/04_honest_forecasting/00_walk_forward_backtest.md`


## How To Use This Notebook
- Work section-by-section; don't skip the markdown.
- Most code cells are incomplete on purpose: replace TODOs and `...`, then run.
- After each section, write 2–4 sentences answering the interpretation prompts (what changed, why it matters).
- Use the **Checkpoint (Self-Check)** section to catch mistakes early.
- Use **Solutions (Reference)** only to unblock yourself; then re-implement without looking.


<a id="environment-bootstrap"></a>
## Environment Bootstrap


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
SAMPLE_DIR = DATA_DIR / 'sample'

PROJECT_ROOT


<a id="single-split"></a>
## The problem with a single split

A single 80/20 train/test split tells you one number: how the model did on a specific 20% window. In a time series, that window is a specific historical regime. If your test window happens to land in a quiet period, the model looks great. If it lands across a financial crisis, it looks terrible. **Neither number is the right answer to 'what error should I expect going forward?'**

Walk-forward backtesting averages over many such test windows, each one strictly later than the data the model was trained on. That gives you a distribution of errors across regimes, which is closer to the actual deployment situation.


### Your Turn (1): Load the macro panel and pick the target


In [ ]:
import pandas as pd
import numpy as np

path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

# Target: next-quarter GDP growth (shift -1 to forecast forward).
y_col = 'y_next'
df[y_col] = df['gdp_growth_qoq'].shift(-1)

# Features: lagged macro indicators (past-only, no leakage).
x_cols = [
    'T10Y2Y_lag1', 'UNRATE_lag1', 'FEDFUNDS_lag1',
    'CPIAUCSL_lag1', 'INDPRO_lag1', 'RSAFS_lag1',
]
x_cols = [c for c in x_cols if c in df.columns]

panel = df[[y_col] + x_cols].dropna().copy()
panel.shape, panel.index.min(), panel.index.max()


### Your Turn (2): Show two single-split RMSEs

Pick two different single splits and fit OLS twice. Confirm the OOS RMSE differs noticeably. The two splits are 1985–2014 / 2015–end and 1985–2010 / 2011–end.


In [ ]:
import statsmodels.api as sm
from src.evaluation import regression_metrics

def fit_predict(train, test, x_cols, y_col):
    Xtr = sm.add_constant(train[x_cols], has_constant='add')
    ytr = train[y_col]
    Xte = sm.add_constant(test[x_cols], has_constant='add')
    res = sm.OLS(ytr, Xtr).fit()
    return res.predict(Xte).to_numpy()

for cutoff in ['2014-12-31', '2010-12-31']:
    train = panel.loc[:cutoff]
    test = panel.loc[pd.Timestamp(cutoff) + pd.Timedelta(days=1):]
    yhat = fit_predict(train, test, x_cols, y_col)
    m = regression_metrics(test[y_col].to_numpy(), yhat)
    print(f'cutoff={cutoff}: rmse={m["rmse"]:.3f}, mae={m["mae"]:.3f}, r2={m["r2"]:.3f}')


Notice that the two RMSEs differ. Neither is wrong; both are facts about specific historical windows. Walk-forward gives you a distribution.


<a id="walk-forward"></a>
## Walk-forward, mechanically

**Expanding window**: training window starts at observation 0 and grows by `step` observations each iteration. The test window is the next `step` observations after the training cutoff. This mimics 'I have all the data I have so far; I refit and predict the next period.'

**Rolling window**: training window has fixed length and slides forward each iteration. Use this if you suspect old data is misleading (regime change). Otherwise expanding is the default.

We use the existing `walk_forward_splits` helper from `src.evaluation`.


### Your Turn (1): Look at the helper


In [ ]:
from src.evaluation import walk_forward_splits

# Initial train window of 40 quarters (10 years) feels right on quarterly data.
# Test window of 4 quarters = one year of forecasts at a time.
n = len(panel)
splits = list(walk_forward_splits(n, initial_train_size=40, test_size=4))
print(f'{len(splits)} splits over {n} observations.')
for s in splits[:3] + splits[-2:]:
    train_dates = panel.index[s.train_slice]
    test_dates = panel.index[s.test_slice]
    print(f'train={train_dates[0].date()}->{train_dates[-1].date()}  test={test_dates[0].date()}->{test_dates[-1].date()}')


<a id="bt-ols"></a>
## Backtest OLS on GDP growth

Loop through the splits, refit OLS on each train window, predict on the test window, collect predictions and realizations.


### Your Turn (1): Implement the OLS walk-forward loop


In [ ]:
def backtest(panel, x_cols, y_col, model_fn, initial_train_size=40, test_size=4):
    """Generic walk-forward backtest. model_fn(X_train, y_train, X_test) -> y_pred."""

    out_idx = []
    out_yhat = []
    out_y = []
    for split in walk_forward_splits(len(panel), initial_train_size=initial_train_size, test_size=test_size):
        train = panel.iloc[split.train_slice]
        test = panel.iloc[split.test_slice]
        yhat = model_fn(train[x_cols], train[y_col], test[x_cols])
        out_idx.extend(test.index)
        out_yhat.extend(yhat)
        out_y.extend(test[y_col].to_numpy())
    return pd.DataFrame({'y_true': out_y, 'y_pred': out_yhat}, index=pd.DatetimeIndex(out_idx))


def ols_model(X_tr, y_tr, X_te):
    Xc_tr = sm.add_constant(X_tr, has_constant='add')
    Xc_te = sm.add_constant(X_te, has_constant='add')
    res = sm.OLS(y_tr, Xc_tr).fit()
    return res.predict(Xc_te).to_numpy()


preds_ols = backtest(panel, x_cols, y_col, ols_model)
preds_ols.tail()


<a id="bt-trees"></a>
## Backtest Random Forest and XGBoost


### Your Turn (1): Random Forest walk-forward


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def rf_model(X_tr, y_tr, X_te, seed=0):
    model = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=seed)
    model.fit(X_tr, y_tr)
    return model.predict(X_te)


preds_rf = backtest(panel, x_cols, y_col, rf_model)
preds_rf.tail()


### Your Turn (2): XGBoost walk-forward


In [ ]:
from xgboost import XGBRegressor

def xgb_model(X_tr, y_tr, X_te, seed=0):
    model = XGBRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.9, random_state=seed, verbosity=0,
    )
    model.fit(X_tr, y_tr)
    return model.predict(X_te)


preds_xgb = backtest(panel, x_cols, y_col, xgb_model)
preds_xgb.tail()


<a id="compare"></a>
## Compare models honestly

Same splits, same windows, same target. The only thing that varied is the model. Now compare aggregate OOS metrics.


### Your Turn (1): Build the comparison table


In [ ]:
rows = []
for name, df_pred in [('OLS', preds_ols), ('RandomForest', preds_rf), ('XGBoost', preds_xgb)]:
    m = regression_metrics(df_pred['y_true'].to_numpy(), df_pred['y_pred'].to_numpy())
    rows.append({'model': name, **m})

results = pd.DataFrame(rows).set_index('model').round(4)
results


### Your Turn (2): Compare to the single-split numbers above

Are the walk-forward RMSEs higher or lower than the single-split RMSEs from the first section? Why?

Write 4–6 sentences. Hint: walk-forward includes early periods when the training window is short and the model has had little to learn from. It also covers more regimes (recessions, expansions, COVID).


In [ ]:
notes = """
...
"""
print(notes)


<a id="errors-over-time"></a>
## Errors over time

An aggregate RMSE smooths over the most interesting question: *when* did each model fail?


### Your Turn (1): Plot rolling absolute errors


In [ ]:
import matplotlib.pyplot as plt

errs = pd.DataFrame({
    'OLS': (preds_ols['y_true'] - preds_ols['y_pred']).abs(),
    'RandomForest': (preds_rf['y_true'] - preds_rf['y_pred']).abs(),
    'XGBoost': (preds_xgb['y_true'] - preds_xgb['y_pred']).abs(),
})
rolling = errs.rolling(8, min_periods=4).mean()

ax = rolling.plot(figsize=(11, 4), title='Rolling 8-quarter mean absolute error')
ax.set_xlabel('quarter')
ax.set_ylabel('|error| (pp)')
plt.tight_layout()


### Your Turn (2): Find the worst quarter for each model


In [ ]:
for name, df_pred in [('OLS', preds_ols), ('RandomForest', preds_rf), ('XGBoost', preds_xgb)]:
    err = (df_pred['y_true'] - df_pred['y_pred']).abs()
    worst = err.idxmax()
    print(f'{name}: worst quarter {worst.date()}, |err|={err.max():.3f}, '
          f'truth={df_pred.loc[worst, "y_true"]:.3f}, predicted={df_pred.loc[worst, "y_pred"]:.3f}')


<a id="ensemble"></a>
## Simple ensemble

When models disagree, the disagreement contains information. The cheapest possible ensemble — average of the three forecasts — often beats every individual model in OOS RMSE.


### Your Turn (1): Build a mean-ensemble forecast


In [ ]:
ens = pd.DataFrame({
    'y_true': preds_ols['y_true'],
    'y_pred': (preds_ols['y_pred'] + preds_rf['y_pred'] + preds_xgb['y_pred']) / 3.0,
})
m = regression_metrics(ens['y_true'].to_numpy(), ens['y_pred'].to_numpy())
print('Ensemble (mean):', {k: round(v, 4) for k, v in m.items()})
print()
print('Individual models for comparison:')
print(results)


### Your Turn (2): Why does the average help?

Write 4–6 sentences. Why does averaging three so-so models often beat the best single one? When does this trick stop working?


In [ ]:
ensemble_notes = """
...
"""
print(ensemble_notes)


<a id="checkpoint-self-check"></a>
## Checkpoint (Self-Check)


In [ ]:
# Sanity asserts (uncomment after the cells above run cleanly):
# assert preds_ols.index.equals(preds_rf.index)
# assert preds_ols.index.equals(preds_xgb.index)
# assert (preds_ols['y_true'] == preds_rf['y_true']).all()
# assert preds_ols.index.is_monotonic_increasing
# assert len(preds_ols) >= 60  # walk-forward should produce many predictions
...


## Extensions (Optional)
- Switch to a **rolling** window (fixed 60-quarter training set, sliding forward). Compare to the expanding-window result.
- Refit at every single quarter (`test_size=1`) instead of every 4. How much does it improve OOS RMSE? How much slower is it?
- Compute a per-decade RMSE (1990s, 2000s, 2010s, 2020s) for each model. Which model is most stable across decades?
- Try a weighted ensemble: weight each model inversely to its rolling RMSE. Does it beat the mean ensemble?


## Reflection
- The walk-forward OOS RMSE is your honest answer to 'how good is this model in deployment.' Would you have shipped the model based on the single-split number?
- The next notebook looks at structural breaks. What in your error-over-time plot suggests there *is* a structural break worth investigating?


<a id="solutions-reference"></a>
## Solutions (Reference)

<details><summary>Solution: Two single-split RMSEs</summary>

The two cutoffs above are the solution. The point is to *see* that two reasonable choices give two different numbers.

</details>

<details><summary>Solution: Walk-forward backtest</summary>

The `backtest()` helper above is the canonical solution. It separates the model from the loop, which lets the same loop test OLS, RF, and XGBoost.

```python
preds_ols = backtest(panel, x_cols, y_col, ols_model)
preds_rf = backtest(panel, x_cols, y_col, rf_model)
preds_xgb = backtest(panel, x_cols, y_col, xgb_model)
```

</details>

<details><summary>Solution: Single-split vs walk-forward</summary>

On the bundled sample you should typically see the walk-forward RMSE come out **higher** than the better of the two single-split RMSEs. That is the honest answer; the single-split number was lucky.

</details>

<details><summary>Solution: Why the mean ensemble helps</summary>

OLS, RF, and XGBoost make systematic errors in different directions. When their errors are partly uncorrelated, averaging cancels noise. The trick stops working when models share the same blind spots — e.g., all three trained on pre-2020 data when COVID arrives.

</details>
